In [2]:
# =====================================================================
# Phase 4: Automated Hyperparameter Tuning with Optuna (LightGBM Edition)
# =====================================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import json
import os
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

# Tắt log rác của cả optuna và lightgbm để nhìn màn hình sạch sẽ
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu và chuẩn bị ma trận số học
df_train = pd.read_csv('../data/processed/train_features.csv')
df_test = pd.read_csv('../data/processed/test_features.csv')

metadata_cols = ['date', 'country_name', 'region', 'income_level', 'vector_disease_risk_score', 'target_log']
synthetic_leakage_cols = ['temperature_celsius', 'precipitation_mm', 'pm25_ugm3']
drop_cols = metadata_cols + synthetic_leakage_cols

features = [col for col in df_train.columns if col not in drop_cols]

X_train_fit = df_train[features].select_dtypes(include=[np.number])
y_train_log = df_train['target_log']
y_train_original = df_train['vector_disease_risk_score']

print(f"🚀 LightGBM sẵn sàng vào lò Tuning: {X_train_fit.shape}")

# 2. Định nghĩa không gian tìm kiếm đặc thù của LightGBM
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=50),
        # LightGBM quản lý độ phức tạp qua num_leaves thay vì max_depth
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        # Ép mô hình lấy ngẫu nhiên dòng/cột để chống overfit chuỗi thời gian
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1 # Tắt cảnh báo trong luồng chạy
    }
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_train_fit):
        X_tr, y_tr = X_train_fit.iloc[train_idx], y_train_log.iloc[train_idx]
        X_va, y_va_orig = X_train_fit.iloc[val_idx], y_train_original.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        
        # Predict & Inverse Log
        preds_log = model.predict(X_va)
        y_pred_original = np.clip(np.expm1(preds_log), 0, 100)
        
        r2 = r2_score(y_va_orig, y_pred_original)
        cv_scores.append(r2)
        
    return np.mean(cv_scores)

# 3. Kích hoạt tối ưu hóa Bayesian
print("🔥 Đang tối ưu hóa LightGBM qua Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, timeout=600)

print("\n🏆 QUÁ TRÌNH TUNING LIGHTGBM HOÀN TẤT!")
print(f"-> Điểm Mean CV R² tốt nhất tìm được: {study.best_value:.4f}")
print("-> Bộ siêu tham số tối ưu LightGBM:")
for k, v in study.best_params.items():
    print(f"   {k}: {v}")

# 4. Xuất file cấu hình tham số
os.makedirs('../metrics', exist_ok=True)

result_to_save = study.best_params.copy()
result_to_save['best_score'] = study.best_value

with open('../metrics/best_params_lightgbm.json', 'w') as f:
    json.dump(result_to_save, f, indent=4)
print("\n✔ Đã lưu tham số tốt nhất tại '../metrics/best_params_lightgbm.json'")

🚀 LightGBM sẵn sàng vào lò Tuning: (10350, 39)
🔥 Đang tối ưu hóa LightGBM qua Optuna...

🏆 QUÁ TRÌNH TUNING LIGHTGBM HOÀN TẤT!
-> Điểm Mean CV R² tốt nhất tìm được: 0.7820
-> Bộ siêu tham số tối ưu LightGBM:
   n_estimators: 200
   num_leaves: 16
   max_depth: 9
   learning_rate: 0.041769681200124316
   bagging_fraction: 0.7435853485550725
   bagging_freq: 2
   feature_fraction: 0.9452894614715269
   min_child_samples: 50
   reg_alpha: 0.03238126901446708
   reg_lambda: 1.386960691861853

✔ Đã lưu tham số tốt nhất tại '../metrics/best_params_lightgbm.json'
